In [ ]:
import csv
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def compute_aliphatic_index(aa_counts, seq_len):
    """
    Compute aliphatic index using the formula from Ikai (1980):
    AI = 100 * (X(Ala) + 2.9*X(Val) + 3.9*(X(Ile) + X(Leu))) / Sequence Length
    """
    if seq_len == 0:
        return 0.0
    ala = aa_counts.get('A', 0)
    val = aa_counts.get('V', 0)
    ile = aa_counts.get('I', 0)
    leu = aa_counts.get('L', 0)
    return 100.0 * (ala + 2.9 * val + 3.9 * (ile + leu)) / seq_len

def analyze_single_protein(record):
    """
    Analyzes a single protein sequence and returns a dictionary of calculated properties.
    """
    sequence = str(record.seq).upper().replace('-', '')
    
    if not sequence:
        print(f"Skipping '{record.id}' as the sequence is empty after removing gaps.")
        return None

    analyzed_seq = ProteinAnalysis(sequence)

    molecular_weight = analyzed_seq.molecular_weight()
    theoretical_pi = analyzed_seq.isoelectric_point()
    charge_at_ph_7_2 = analyzed_seq.charge_at_pH(7.2)
    ext_coeff_cystine, ext_coeff_reduced = analyzed_seq.molar_extinction_coefficient()
    instability_index = analyzed_seq.instability_index()
    stability = "Stable" if instability_index < 40 else "Unstable"
    aromaticity = analyzed_seq.aromaticity()
    gravy = analyzed_seq.gravy(scale="Cowan7.5")
    amino_acid_counts = analyzed_seq.count_amino_acids()

    # Flexibility
    try:
        flexibility_scores = analyzed_seq.flexibility()
        average_flexibility = sum(flexibility_scores) / len(flexibility_scores) if flexibility_scores else 0.0
    except Exception:
        average_flexibility = 0.0

    # Secondary structure
    try:
        alpha_helix_fraction, beta_turn_fraction, random_coil_fraction = analyzed_seq.secondary_structure_fraction()
    except Exception:
        alpha_helix_fraction = beta_turn_fraction = random_coil_fraction = 0.0

    # Charged residue counts (E, D are negative; K, R, H are positive)
    negative_count = amino_acid_counts.get('D', 0) + amino_acid_counts.get('E', 0)
    positive_count = amino_acid_counts.get('K', 0) + amino_acid_counts.get('R', 0) + amino_acid_counts.get('H', 0)

    # Custom Aliphatic Index
    aliphatic_index = compute_aliphatic_index(amino_acid_counts, len(sequence))

    results = {
        "Protein ID": record.id,
        "Sequence Length": len(sequence),
        "Molecular Weight": f"{molecular_weight:.2f}",
        "Theoretical pI": f"{theoretical_pi:.2f}",
        "Total Negatively Charged Residues": negative_count,
        "Total Positively Charged Residues": positive_count,
        "charge": f"{charge_at_ph_7_2:.2f}",
        "Ext. Coeff (Cystines)": ext_coeff_cystine,
        "Ext. Coeff (Reduced Cys)": ext_coeff_reduced,
        "Instability Index": f"{instability_index:.2f}",
        "Stability": stability,
        "Aliphatic Index": f"{aliphatic_index:.2f}",
        "aromaticity": f"{aromaticity:.2f}",
        "GRAVY": f"{gravy:.3f}",
        "Average Flexibility": f"{average_flexibility:.2f}",
        "% Alpha Helix": f"{alpha_helix_fraction * 100:.2f}",
        "% Beta Turn": f"{beta_turn_fraction * 100:.2f}",
        "% Random Coil": f"{random_coil_fraction * 100:.2f}",
    }

    # Add individual amino acid counts
    for aa in "ACDEFGHIKLMNPQRSTVWY":
        results[f"{aa} Count"] = amino_acid_counts.get(aa, 0)

    return results

def analyze_protein_from_fasta(fasta_file, output_csv):
    """
    Reads a FASTA file, analyzes each protein, and saves the results to a CSV file.
    """
    try:
        with open(fasta_file, "r") as f:
            pass
    except FileNotFoundError:
        print(f"Error: The FASTA file '{fasta_file}' was not found.")
        return
    except Exception as e:
        print(f"Error reading FASTA file '{fasta_file}': {e}")
        return

    header = [
        "Protein ID", "Sequence Length", "Molecular Weight", "Theoretical pI",
        "Total Negatively Charged Residues", "Total Positively Charged Residues", "charge",
        "Ext. Coeff (Cystines)", "Ext. Coeff (Reduced Cys)",
        "Instability Index", "Stability", "Aliphatic Index", "aromaticity", "GRAVY",
        "Average Flexibility", "% Alpha Helix", "% Beta Turn", "% Random Coil"
    ] + [f"{aa} Count" for aa in "ACDEFGHIKLMNPQRSTVWY"]

    with open(output_csv, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(header)

        for record in SeqIO.parse(fasta_file, "fasta"):
            try:
                results = analyze_single_protein(record)
                if results is None:
                    continue
                row = [results.get(key, "") for key in header]
                writer.writerow(row)
            except Exception as e:
                print(f"Skipping '{record.id}' due to error: {e}")

    print(f"Analysis complete. Results saved to '{output_csv}'")


# --- Main execution block ---
if __name__ == "__main__":
    fasta_file_path = "mix_align_trim_3_helix.fasta"
    output_csv_path = "mix_align_trim_3_helix.csv"
    
    analyze_protein_from_fasta(fasta_file_path, output_csv_path)


Analysis complete. Results saved to 'Helix-13/final_psy.csv'
